# Day 9: Snowflake Security, Dynamic Masking & dbt Gold Marts

This notebook simulates Gold metric calculations (30-day readmissions) and role-based column masking using DuckDB and Pandas.

In [ ]:
import pandas as pd
import duckdb

con = duckdb.connect(database=':memory:')

# Create sample silver data
con.execute("""
CREATE TABLE condition_occurrence (
    condition_occurrence_id VARCHAR,
    person_id VARCHAR,
    condition_start_date DATE
);

INSERT INTO condition_occurrence VALUES 
('C1', 'P1', '2023-01-01'),
('C2', 'P1', '2023-01-15'), -- 14 day readmit
('C3', 'P2', '2023-02-01'),
('C4', 'P2', '2023-04-01'); -- 60 day readmit (no 30-day)
""")

print("--- SILVER OMOP CONDITIONS ---")
print(con.execute("SELECT * FROM condition_occurrence").fetchdf())


In [ ]:
# Simulate dbt Gold Model: 30-Day Readmissions
gold_sql = """
WITH readmission_calc AS (
    SELECT 
        a.person_id,
        a.condition_occurrence_id AS index_admission_id,
        a.condition_start_date AS index_date,
        b.condition_occurrence_id AS readmission_id,
        b.condition_start_date AS readmission_date,
        date_diff('day', a.condition_start_date, b.condition_start_date) as days_to_readmit
    FROM condition_occurrence a
    JOIN condition_occurrence b 
      ON a.person_id = b.person_id
      AND a.condition_occurrence_id != b.condition_occurrence_id
      AND b.condition_start_date > a.condition_start_date
      AND b.condition_start_date <= a.condition_start_date + INTERVAL 30 DAY
)
SELECT 
    person_id,
    COUNT(DISTINCT index_admission_id) as total_index_admissions,
    COUNT(DISTINCT readmission_id) as total_30_day_readmissions
FROM readmission_calc
GROUP BY person_id
"""

con.execute(f"CREATE TABLE readmissions_30_day AS {gold_sql}")

print("\n--- GOLD 30-DAY READMISSIONS MART ---")
print(con.execute("SELECT * FROM readmissions_30_day").fetchdf())


In [ ]:
# Simulate Dynamic Masking (Role-based access via Pandas wrapping)
def query_with_role(query: str, role: str) -> pd.DataFrame:
    df = con.execute(query).fetchdf()
    if role == 'DATA_ANALYST':
        # Mask person_id for analysts
        if 'person_id' in df.columns:
            df['person_id'] = 'REDACTED'
    return df

print("\n--- QUERY AS DATA_ENGINEER ---")
print(query_with_role("SELECT * FROM readmissions_30_day", 'DATA_ENGINEER'))

print("\n--- QUERY AS DATA_ANALYST (Simulated Masking) ---")
print(query_with_role("SELECT * FROM readmissions_30_day", 'DATA_ANALYST'))
